# OpenAI Agents SDK / Swarm

**Domain:** Agentic AI  ·  **recommended addition**  ·  **runnable:** yes  ·  _needs API key for the live cell_

A refresher on **OpenAI's first-party agent stack** — the lineage from **Swarm** (the tiny, educational, now-archived experiment) to the **OpenAI Agents SDK** (`openai-agents`, the production successor). The whole stack is built on two primitives: an **Agent** (an LLM with instructions + tools) and a **handoff** (one agent passing control of the conversation to another). That's the entire multi-agent model — no graphs, no message buses, just agents transferring the baton.

If you know the bare [[react]] loop, this is that loop wrapped as a product, plus a first-class notion of *routing between specialists*. Compare with [[langchain]]/[[langgraph]], [[crewai]], [[autogen]], [[smolagents]], and [[llamaindex-agents]] for the same idea in other framings.

## 1. What & Why

**Swarm** was OpenAI's late-2024 *experimental, educational* sketch of multi-agent orchestration: a few hundred lines showing that you don't need a heavyweight framework to coordinate agents — you need just **Agents** and **handoffs**. It was deliberately stateless, ran entirely client-side on top of Chat Completions, and was explicitly **not for production**. OpenAI has since **archived Swarm** and folded its ideas into the **OpenAI Agents SDK** (the `openai-agents` package), the supported, production-ready evolution.

**The problem it solves.** A single prompt with twenty tools becomes unreliable — the model picks wrong tools, the instructions sprawl, and you can't reason about its behaviour. The fix mirrors how humans organize work: build **several small, sharply-scoped agents** (a triage agent, a refunds agent, a sales agent) and let them **hand off** to one another. Each agent has a short instruction set and only the tools it needs; the conversation moves to whichever specialist is relevant. The result is far more steerable than one mega-agent.

**What the Agents SDK adds on top of Swarm's two primitives:**

- **Tools** — any Python function becomes a tool via `@function_tool` (signature + docstring → JSON schema the model sees). It also ships hosted tools (web search, file search, code interpreter).
- **Handoffs** — a first-class mechanism (`handoffs=[...]`) for delegating to another agent; under the hood a handoff is exposed to the model as a tool call.
- **Guardrails** — input/output validators that run in parallel and can halt a run early (e.g. reject off-topic or unsafe requests).
- **Sessions** — automatic conversation-history management across turns, so you don't thread messages by hand.
- **Tracing** — built-in run traces you can view in the OpenAI dashboard or export to other observability tools.

**When to reach for it.** You're on OpenAI models and want a *minimal, official* agent loop with clean multi-agent routing and tracing, without adopting a large framework. **When not to.** If one agent with a couple of tools answers every request, skip the multi-agent machinery. If you need explicit, persisted state-machine control or heavy cross-provider orchestration, [[langgraph]] fits better. And never start a new project on **Swarm itself** — it's archived; use the Agents SDK.

## 2. Mental Model

**A team of specialists passing a baton.** Each agent is one teammate with a job description (instructions) and a toolbox. A **handoff** is the baton pass: the current agent decides "this isn't mine — it's a refund," and hands the *whole live conversation* to the Refunds agent, who continues from there with its own instructions and tools. There is no orchestrator sitting above them; control simply *is* whichever agent currently holds the baton.

```
                user: "I want a refund for order A123"
                                │
                                ▼
                    ┌────────────────────┐
                    │   TRIAGE agent     │  tools: (handoff → Refunds, Sales)
                    │  instr: route me   │
                    └─────────┬──────────┘
                              │  handoff (a tool call that
                              │  returns the next agent)
                              ▼
                    ┌────────────────────┐
                    │   REFUNDS agent    │  tools: issue_refund, lookup_order
                    │  instr: do refunds │
                    └─────────┬──────────┘
                              │  issue_refund("A123")
                              ▼
                      "Refund issued for order A123."
```

The crucial trick: **a handoff is just a special tool call.** The model is offered `transfer_to_refunds` alongside its normal tools; when it "calls" that, the runner swaps the active agent and keeps looping. So the same Thought→Action→Observation [[react]] loop drives everything — sometimes the Action is a normal tool, sometimes it's "become a different agent." That single idea is the whole framework.

## 3. Key Concepts

| Concept | What it is |
|---|---|
| **Agent** | An LLM configured with `instructions` (its system prompt / role), a list of `tools`, optional `handoffs`, an `output_type`, and a model. The unit you compose everything from. |
| **`Runner`** | The executor that runs the agent loop. `Runner.run(...)` (async), `Runner.run_sync(...)` (blocking), `Runner.run_streamed(...)` (token/event stream). It calls the model, executes tool calls, applies handoffs, and stops at a final output. |
| **Tool (`@function_tool`)** | Decorator that turns a Python function into a tool — it reads the **signature, type hints, and docstring** to build the JSON schema the model sees. Also: hosted tools (web/file search, code interpreter) and agents-as-tools. |
| **Handoff** | Delegation to another agent. Declared via `handoffs=[other_agent]` (or `handoff(agent, ...)` for customization). Surfaced to the model as a `transfer_to_<name>` tool; calling it transfers control **and** the conversation. |
| **Guardrail** | A validator on agent input or output that runs alongside the agent and can raise a tripwire to abort the run early — e.g. block prompt injection, enforce a topic, validate structured output. |
| **Session** | Built-in conversation memory (e.g. `SQLiteSession`) that persists message history across `Runner.run` calls, so multi-turn chats don't require manual history threading. |
| **`output_type`** | Set it to a Pydantic model / dataclass and the agent returns **validated structured output** instead of free text. |
| **Tracing** | Automatic spans for every run (model calls, tool calls, handoffs, guardrails), viewable in the OpenAI traces dashboard or exportable to external tracers. |
| **`RunResult`** | What a run returns: `.final_output` (the answer), the full message history, the last agent, tool/handoff events. |

## 4. Setup

The modern package is **`openai-agents`** (imported as `agents`). The old experiment was **`swarm`** (install from its archived GitHub repo) — use it only to read, not to build.

```bash
pip install openai-agents          # the supported SDK; `import agents`
export OPENAI_API_KEY="sk-..."     # the Runner needs this to call a model

# Legacy / read-only — Swarm is archived and not on PyPI under this name:
# pip install git+https://github.com/openai/swarm.git
```

Minimal live program:

```python
from agents import Agent, Runner
agent = Agent(name="Assistant", instructions="You are concise and helpful.")
print(Runner.run_sync(agent, "Say hello in five words.").final_output)
```

**Examples 1 and 2 below are pure-Python models of the SDK's mechanics** — tool-schema generation and the handoff loop — so they execute in a fresh kernel with **no install and no API key**. **Example 3** drives the real SDK and is gated behind an install + key check. The cell below reports what's available.

In [ ]:
# Environment check — Examples 1 & 2 run offline; Example 3 needs the SDK + a key.
import importlib.util as u, os

def _installed(name: str) -> bool:
    try:
        return u.find_spec(name) is not None
    except ModuleNotFoundError:   # a missing parent package raises rather than returns None
        return False

has_agents = _installed("agents")        # the `openai-agents` package
has_openai = _installed("openai")
has_key    = bool(os.getenv("OPENAI_API_KEY"))

print("openai-agents installed :", has_agents)
print("openai installed        :", has_openai)
print("OPENAI_API_KEY present  :", has_key)
print()
print("Examples 1 & 2 are pure-Python models of the SDK's mechanics — they run")
print("offline, no key, no install. Example 3 drives the real SDK and is gated")
print("behind the checks above; without them it prints the canonical call shape.")

## 5. Worked Examples

### Example 1 — A function becomes a tool

Everything starts with a **tool**, and the SDK's `@function_tool` builds one by **introspecting your function**: the name comes from `__name__`, the description from the docstring, and the argument schema from the type hints. That JSON schema is exactly what the model sees when it decides whether and how to call the tool — which is why clear docstrings and real type hints matter. The cell below reproduces that derivation with the standard library alone, so you can see the artifact the SDK hands to the model. Parameters without a default land in `required`; ones with a default don't.

In [ ]:
import inspect, json
from typing import get_type_hints

_PY_TO_JSON = {int: "integer", float: "number", str: "string", bool: "boolean"}

def build_tool_schema(fn):
    """Mirror what @function_tool does: derive name, description, and a JSON
    parameter schema from a function's signature, type hints, and docstring."""
    hints = get_type_hints(fn)
    props, required = {}, []
    for pname, param in inspect.signature(fn).parameters.items():
        props[pname] = {"type": _PY_TO_JSON.get(hints.get(pname, str), "string")}
        if param.default is inspect.Parameter.empty:
            required.append(pname)
    return {
        "name": fn.__name__,
        "description": (fn.__doc__ or "").strip().splitlines()[0],
        "parameters": {"type": "object", "properties": props, "required": required},
    }

def get_weather(city: str, units: str = "celsius") -> str:
    """Get the current weather for a city."""
    return f"18 {units} and clear in {city}"

# This dict is what the model receives to reason about the tool:
print(json.dumps(build_tool_schema(get_weather), indent=2))

### Example 2 — Handoffs: the relay loop made explicit

This is the defining concept. A `Runner` needs a real LLM to choose actions, so to *run* the loop offline we stand in a deterministic planner for the model. The point is to show the exact mechanic the SDK implements: **a handoff is a tool call that returns another agent.** When the active agent "calls" `transfer_to_refunds`, the runner sees the result is an `Agent`, swaps the active agent, and keeps looping — the conversation is now the Refunds agent's. A triage agent routes a refund request to a refunds specialist, which then runs its own tool. Normal tools return data; handoff tools return an agent. Same loop handles both.

In [ ]:
from dataclasses import dataclass, field

@dataclass
class Agent:
    name: str
    instructions: str
    tools: dict = field(default_factory=dict)   # tool name -> callable

# A plain tool returns data; a *handoff* tool returns another Agent.
def issue_refund(order_id: str) -> str:
    return f"Refund issued for order {order_id}."

refunds_agent = Agent(
    name="Refunds",
    instructions="Process refunds. Call issue_refund(order_id).",
    tools={"issue_refund": issue_refund},
)

def transfer_to_refunds() -> Agent:
    """Hand the conversation off to the Refunds agent."""
    return refunds_agent

triage_agent = Agent(
    name="Triage",
    instructions="Route the user. For refund requests, call transfer_to_refunds().",
    tools={"transfer_to_refunds": transfer_to_refunds},
)

def planner(agent, state):
    """Stand-in for the LLM: pick the next tool call for the active agent.
    The real SDK gets this decision from the model's tool-calling output."""
    if agent.name == "Triage":
        return ("transfer_to_refunds", {})
    if agent.name == "Refunds" and "refunded" not in state:
        return ("issue_refund", {"order_id": state["order_id"]})
    return None  # nothing left to do -> final answer

def run(agent, state, max_turns=8):
    for turn in range(1, max_turns + 1):
        decision = planner(agent, state)
        if decision is None:
            return f"[{agent.name}] done — it held the baton after the handoff."
        tool_name, args = decision
        result = agent.tools[tool_name](**args)
        if isinstance(result, Agent):                       # <- the handoff
            print(f"turn {turn}: {agent.name} --handoff--> {result.name}")
            agent = result
        else:                                               # <- a normal tool
            print(f"turn {turn}: {agent.name} called {tool_name}{tuple(args.values())} -> {result}")
            state["refunded"] = True
    return "stopped: hit max_turns"

print(run(triage_agent, {"order_id": "A123"}))

### Example 3 — The real Agents SDK (gated)

With a real model the planner disappears: you declare the agents, list the handoff, and call `Runner.run_sync`. The model itself decides to transfer to Refunds and then call the tool. This cell runs only if `openai-agents` is installed **and** `OPENAI_API_KEY` is set; otherwise it prints the canonical call shape so the notebook still executes top-to-bottom. Note how little code the production version is — the framework is genuinely thin.

In [ ]:
if has_agents and has_key:
    from agents import Agent as SDKAgent, Runner, function_tool

    @function_tool
    def issue_refund(order_id: str) -> str:
        """Issue a refund for the given order id."""
        return f"Refund issued for order {order_id}."

    refunds = SDKAgent(
        name="Refunds",
        instructions="You process refunds. Use the issue_refund tool, then confirm.",
        tools=[issue_refund],
    )
    triage = SDKAgent(
        name="Triage",
        instructions="Route the user. Hand off to Refunds for any refund request.",
        handoffs=[refunds],
    )
    result = Runner.run_sync(triage, "I want a refund for order A123.")
    print(result.final_output)
else:
    print("Skipping live SDK run (need `pip install openai-agents` + OPENAI_API_KEY).")
    print("With both set, the call shape is:")
    print()
    print("    from agents import Agent, Runner, function_tool")
    print("    @function_tool")
    print("    def issue_refund(order_id: str) -> str: ...")
    print("    refunds = Agent(name='Refunds', instructions=..., tools=[issue_refund])")
    print("    triage  = Agent(name='Triage', instructions=..., handoffs=[refunds])")
    print("    result  = Runner.run_sync(triage, 'I want a refund for order A123.')")
    print("    print(result.final_output)   # -> 'Refund issued for order A123.'")

## 6. Gotchas & Pitfalls

- **Starting a new project on Swarm.** Swarm is **archived and educational** — no support, no guarantees. Read it to understand the model, then build on the **OpenAI Agents SDK** (`openai-agents`). Tutorials referencing `from swarm import Swarm` are outdated.
- **Confusing handoffs with agents-as-tools.** A **handoff** *transfers control* — the target agent takes over the conversation and the original agent is done. **Agent-as-tool** *calls a sub-agent and returns its result* to the caller, which keeps control. Use a handoff for "this belongs to someone else," a tool-agent for "go fetch me this." Picking the wrong one produces baffling control flow.
- **Weak tool/handoff descriptions.** The model routes purely on names and descriptions. A handoff named `transfer` with no description, or tools with no docstrings/type hints, gives the model nothing to route on — it picks wrong or loops. The docstring *is* the tool's prompt.
- **Forgetting the runner is async under the hood.** `Runner.run(...)` is a coroutine; use `await` in async code or `Runner.run_sync(...)` in a script/notebook top level. Calling `run` without awaiting it silently does nothing.
- **No turn/cost ceiling.** Like any [[react]] loop, a confused set of agents can ping-pong handoffs or call tools indefinitely. Set `max_turns` and watch token spend; every turn is a model call.
- **Losing the conversation across turns.** A bare `Runner.run` doesn't remember the last turn. For multi-turn chat, pass a **Session** (e.g. `SQLiteSession`) or feed `result.to_input_list()` back in — otherwise each call starts cold.
- **Over-fragmenting into too many agents.** Multi-agent is a tool for *separation of concerns*, not a default. Three crisp agents beat ten overlapping ones; if a single agent with a few tools does the job, don't add handoffs.
- **Assuming it's model-agnostic by default.** The SDK is OpenAI-first. It can target other providers via the Chat Completions interface / LiteLLM, but hosted tools and some features assume OpenAI models — check before betting on a non-OpenAI backend.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-offs vs the Agents SDK |
|---|---|---|
| **OpenAI Agents SDK (this)** | Minimal, official agent loop on OpenAI models; clean multi-agent **handoffs**, guardrails, sessions, built-in tracing | Thin and unopinionated; OpenAI-centric; younger ecosystem than LangChain |
| **Swarm (archived)** | *Reading* to learn the Agent+handoff idea | Experimental, unsupported, no production features — don't build on it; the SDK is its successor |
| **Raw [[react]] loop / Chat Completions + tools** | A tiny single-file agent; full control | You hand-build tool dispatch, handoffs, memory, tracing. The SDK gives all that for ~the same code |
| **[[langgraph]]** | Explicit **state-machine** control, branching, persistence, human-in-the-loop checkpoints | More powerful and provider-agnostic, but heavier and more boilerplate; the SDK is simpler when handoffs suffice |
| **[[langchain]]** | Huge integration catalog, off-the-shelf chains/tools | Broader but heavier and more abstracted; the SDK is leaner and OpenAI-native |
| **[[crewai]] / [[autogen]]** | Role-based crews / conversational multi-agent as the primary abstraction | Richer multi-agent framing and patterns; the SDK models the same with bare handoffs and less ceremony |
| **[[smolagents]] / [[llamaindex-agents]]** | Code-action agents / RAG-centric agents over your own data | Specialized strengths (code execution; `QueryEngineTool` RAG) the general-purpose SDK doesn't focus on |

**Rule of thumb:** if you're on OpenAI models and want the *smallest* official path to a tool-using, multi-specialist agent with tracing, reach for the Agents SDK. Step up to [[langgraph]] when you need explicit persisted state machines or strict control; step out to a multi-agent framework when roles/conversations are your central abstraction; and only *read* Swarm — never ship it.

## 8. Resources

- **OpenAI Agents SDK — official docs** — https://openai.github.io/openai-agents-python/
- **OpenAI Agents SDK — GitHub (source, examples)** — https://github.com/openai/openai-agents-python
- **Handoffs guide (the multi-agent primitive)** — https://openai.github.io/openai-agents-python/handoffs/
- **Tools & `@function_tool` guide** — https://openai.github.io/openai-agents-python/tools/
- **Swarm — archived original (read to learn the model)** — https://github.com/openai/swarm
- **OpenAI — "A practical guide to building agents" (PDF)** — https://cdn.openai.com/business-guides-and-resources/a-practical-guide-to-building-agents.pdf
- **ReAct prompting (the loop underneath the runner)** — https://arxiv.org/abs/2210.03629

YOUR ANSWER HERE

YOUR ANSWER HERE

YOUR ANSWER HERE

In [ ]:
class Agent:
    ...


def run(agent, message, planner, session=None, guardrails=(), max_turns=8):
    ...
# YOUR CODE HERE
raise NotImplementedError()

YOUR ANSWER HERE

YOUR ANSWER HERE